In [2]:
import pandas as pd

In [3]:
combined = pd.read_csv('../data/rentora_combined_raw.csv')

C:\Users\KUNDAN KUMAR\AppData\Local\Temp\ipykernel_45584\1376320429.py:1: DtypeWarning: Columns (0: size_sqft) have mixed types. Specify dtype option on import or set low_memory=False.
  combined = pd.read_csv('../data/rentora_combined_raw.csv')


In [4]:
combined.head()

,city,locality,bhk,rent,size_sqft,furnishing,bathrooms,latitude,longitude,source
0,Ahmedabad,Bodakdev,2.0,20000.0,1450.0,Furnished,2.0,23.044592,72.517344,makaan
1,Ahmedabad,CG Road,1.0,7350.0,210.0,Semi-Furnished,1.0,23.026011,72.556718,makaan
2,Ahmedabad,Jodhpur,3.0,22000.0,1900.0,Unfurnished,3.0,23.016927,72.520432,makaan
3,Ahmedabad,Sanand,2.0,13000.0,1285.0,Semi-Furnished,2.0,23.023888,72.385148,makaan
4,Ahmedabad,Navrangpura,2.0,18000.0,1600.0,Furnished,2.0,23.036000,72.564343,makaan


In [9]:
#This shows the total number of rows and columns in the dataset. It helps verify that the merge was successful and no unexpected records were lost.
combined.shape

(206921, 10)

In [10]:
#This provides the number of non-null values, column names, and data types. It helps identify missing values and whether each column has the correct datatype.
combined.info()

<class 'pandas.DataFrame'>
RangeIndex: 206921 entries, 0 to 206920
Data columns (total 10 columns):
 #   Column      Non-Null Count   Dtype  
---  ------      --------------   -----  
 0   city        206921 non-null  str    
 1   locality    206921 non-null  str    
 2   bhk         206921 non-null  float64
 3   rent        206921 non-null  float64
 4   size_sqft   206921 non-null  object 
 5   furnishing  206921 non-null  str    
 6   bathrooms   206865 non-null  float64
 7   latitude    206921 non-null  float64
 8   longitude   206921 non-null  float64
 9   source      206921 non-null  str    
dtypes: float64(5), object(1), str(4)
memory usage: 15.8+ MB


In [11]:
#This confirms that numerical columns are stored as numeric types and categorical columns are stored as object/string types.
combined.dtypes

city              str
locality          str
bhk           float64
rent          float64
size_sqft      object
furnishing        str
bathrooms     float64
latitude      float64
longitude     float64
source            str
dtype: object

In [12]:
#This identifies columns containing missing values. Based on the number of missing entries, we can decide whether to remove rows, fill the values, or keep them.
# Count missing values in every column
combined.isnull().sum()

city           0
locality       0
bhk            0
rent           0
size_sqft      0
furnishing     0
bathrooms     56
latitude       0
longitude      0
source         0
dtype: int64

In [13]:
# Option A: fill with the median bathrooms for that BHK size (sensible default)
combined['bathrooms'] = combined['bathrooms'].fillna(
    combined.groupby('bhk')['bathrooms'].transform('median')
)

In [14]:
combined.isnull().sum()

city          0
locality      0
bhk           0
rent          0
size_sqft     0
furnishing    0
bathrooms     0
latitude      0
longitude     0
source        0
dtype: int64

In [15]:
# Number of duplicate rows

#Duplicate records can bias analysis and machine learning models. This check tells us whether duplicate entries exist.
combined.duplicated().sum()

np.int64(89360)

In [16]:
#This tells you whether duplicates are coming mostly from Makaan (more likely, given its scrape-based origin and larger size) or spread across both sources evenly.
print(combined['source'].value_counts())
print(combined[combined.duplicated()]['source'].value_counts())

source
makaan       193011
april2024     13910
Name: count, dtype: int64
source
makaan       86643
april2024     2717
Name: count, dtype: int64


In [17]:
#Confirms the pattern: 45% of Makaan's rows are duplicates (86,643 of 193,011) vs. ~20% of April2024's (2,717 of 13,910). Makaan being much more scrape-heavy at 200K rows makes this expected — same listings getting picked up multiple times during scraping.
#dropping duplicates is a good idea to avoid biasing the analysis and models.
combined = combined.drop_duplicates()
print("New shape after removing duplicates:", combined.shape)

New shape after removing duplicates: (117561, 10)


In [18]:
#This will show min/max/mean/percentiles for each — what we're checking for is anything obviously broken: a rent minimum of ₹0 or ₹100, a size_sqft of 1, a bhk of 50, etc. Scraped data almost always has a handful of these outlier/junk entries.
combined[['rent', 'size_sqft', 'bhk', 'bathrooms']].describe()

,rent,bhk,bathrooms
count,1.175610e+05,117561.000000,117561.000000
mean,5.239012e+04,2.126658,2.095669
std,1.156629e+05,0.997778,0.975216
min,1.200000e+03,1.000000,1.000000
25%,1.300000e+04,1.000000,1.000000
50%,2.200000e+04,2.000000,2.000000
75%,4.000000e+04,3.000000,3.000000
max,5.885000e+06,15.000000,19.000000


In [19]:
#Next: quantify how many rows are actually extreme, before deciding what to do
print("Rent > 5,00,000:", (combined['rent'] > 500000).sum())
print("Rent > 10,00,000:", (combined['rent'] > 1000000).sum())
print("BHK > 8:", (combined['bhk'] > 8).sum())
print("Bathrooms > 10:", (combined['bathrooms'] > 10).sum())

# look at a few actual extreme rent rows
combined[combined['rent'] > 500000][['city', 'locality', 'rent', 'bhk', 'size_sqft']].head(10)

Rent > 5,00,000: 1435
Rent > 10,00,000: 276
BHK > 8: 101
Bathrooms > 10: 20


,city,locality,rent,bhk,size_sqft
30234,Bangalore,Ashok Nagar,800000.0,4.0,7000.0
31370,Bangalore,Koramangala,900000.0,10.0,9500.0
38681,Bangalore,Krishnarajapura,600000.0,2.0,1600.0
42309,Chennai,Teynampet,600000.0,6.0,6000.0
51030,Delhi,Hauz Khas,600000.0,5.0,7000.0
51031,Delhi,Jor bagh,900000.0,4.0,3550.0
51032,Delhi,Connaught Place,1000000.0,5.0,5000.0
51829,Delhi,Vasant Kunj,750000.0,6.0,10000.0
51830,Delhi,West End,550000.0,4.0,4000.0
51831,Delhi,DLF Farms,600000.0,5.0,9000.0


In [20]:
#This isolates just the handful of truly extreme rows (above ₹20L) — that's where actual data errors are most likely hiding (e.g. a sale price entered as rent, or a stray extra zero).
combined[combined['rent'] > 2000000][['city', 'locality', 'rent', 'bhk', 'size_sqft']].sort_values('rent', ascending=False).head(10)

,city,locality,rent,bhk,size_sqft
70682,Delhi,Aurungzeb Road,5885000.0,12.0,15562.0
72072,Delhi,Tilak Marg,4129000.0,15.0,17211.0
72922,Delhi,India Gate,4011000.0,15.0,15515.0
72312,Delhi,Tilak Marg,4010000.0,8.0,11010.0
72923,Delhi,Tilak Marg,3746000.0,10.0,16521.0
71390,Delhi,Prithviraj Road,3624000.0,15.0,18521.0
77767,Delhi,Chanakya Puri,3500000.0,6.0,11000.0
79878,Delhi,Chanakya Puri,3500000.0,12.0,19800.0
197581,Delhi,Amrita Shergill Marg,3010101.0,12.0,"14,521 sq ft"
197578,Delhi,Aurungzeb Road,2491184.0,5.0,"6,521 sq ft"


In [21]:
#Percentile Based:-Rather than picking an arbitrary threshold and potentially cutting real luxury listings, use statistics to find genuine outliers:
q99 = combined['rent'].quantile(0.999)
print(f"99.9th percentile rent: {q99}")
print("Rows above it:", (combined['rent'] > q99).sum())

99.9th percentile rent: 1282079.6400000004
Rows above it: 118
